# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muneeb-th/ML-Assignment-1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/Muneeb-th/ML-Assignment-1/main/data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
print(f"Loaded {len(df)} rows")

Loaded 30000 rows


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule: A page is worth reviewing if it's stale (not updated in 6+
months), still getting meaningful traffic, and either declining or
showing weak CTR for its position. Signal check: staleness is behind
FlyRank's stale_visible_page flag; low CTR-at-position is behind the
CTR-fix logic.

In [11]:
import numpy as np

# Signal checks first
df["is_stale"] = (df["days_since_last_update"] >= 180).astype(int)
stale_bucket = df.groupby("is_stale")["is_declining"].agg(["mean", "count"])
print("Signal A - Staleness bucket (n and decline rate):")
print(stale_bucket)

df["median_ctr_for_tier"] = df.groupby("position_tier")["ctr"].transform("median")
df["is_low_ctr"] = (df["ctr"] < df["median_ctr_for_tier"]).astype(int)
ctr_bucket = df.groupby("is_low_ctr")["is_declining"].agg(["mean", "count"])
print("\nSignal B - Low CTR bucket (n and decline rate):")
print(ctr_bucket)

# Encode the rule as a transparent score
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["score"] = stale * visible * df["impressions_90d"]  # readable on purpose

# Reason codes
df["reason_code"] = np.select(
    [
        (df["is_stale"] == 1) & (visible == 1) & (df["is_declining"] == 1),
        (df["is_stale"] == 1) & (visible == 1),
        (df["is_low_ctr"] == 1) & (visible == 1),
    ],
    ["stale_and_declining", "stale_but_visible", "low_ctr_at_position"],
    default="no_flag"
)

# Action label
df["action"] = np.where(df["score"] > 0, "review_for_refresh", "monitor")

print(f"\nBase rate (declining share overall): {df['is_declining'].mean():.3f}")

Signal A - Staleness bucket (n and decline rate):
              mean  count
is_stale                 
0         0.542480  29826
1         0.471264    174

Signal B - Low CTR bucket (n and decline rate):
                mean  count
is_low_ctr                 
0           0.502630  16921
1           0.593088  13079

Base rate (declining share overall): 0.542


Signal A (staleness) — OPPOSITE: stale pages actually show a LOWER
decline rate (47.1%) than non-stale pages (54.3%). This contradicts the
assumption that staleness predicts decline.

Signal B (low CTR at position) — MIXED: low-CTR pages show a decline
rate of 59.3% vs 58.3% for others — barely above the 54.2% base rate.
Not a meaningfully useful signal on its own.

Given both checks, I'm keeping staleness + visibility in my rule as a
volume/visibility filter rather than a decline predictor — it identifies
pages worth human attention regardless of whether they're currently
declining, which matches the review-queue framing better than trying to
predict decline directly.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
import os
os.makedirs("work/outputs", exist_ok=True)

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)
ranked[["content_id", "score", "reason_code", "action", "is_declining",
        "impressions_90d", "days_since_last_update", "ctr"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False
)
print(f"Wrote {len(ranked)} rows to work/outputs/baseline_action_score.csv")
print(ranked[["content_id", "score", "reason_code", "action"]].head(20))

Wrote 30000 rows to work/outputs/baseline_action_score.csv
              content_id  score          reason_code              action
0   content_cf56e2e2e282  61678  stale_and_declining  review_for_refresh
1   content_7368877ea310  59472  stale_and_declining  review_for_refresh
2   content_1bfaa38ff26c  25715  stale_and_declining  review_for_refresh
3   content_0a91db491d14  13299  stale_and_declining  review_for_refresh
4   content_5feee3994adb   7812  stale_and_declining  review_for_refresh
5   content_c2d929d83eaa   7558  stale_and_declining  review_for_refresh
6   content_b16bd7307b39   4590  stale_and_declining  review_for_refresh
7   content_fe16a55cd13d   4556  stale_and_declining  review_for_refresh
8   content_ecb6215e79fd   4429  stale_and_declining  review_for_refresh
9   content_928af3e22c80   1697  stale_and_declining  review_for_refresh
10  content_e3ff1b093148   1408  stale_and_declining  review_for_refresh
11  content_bdbec75c1148   1316    stale_but_visible  review_for_

In [13]:
top20 = ranked.head(20)[["content_id", "score", "reason_code", "action",
                          "is_declining", "impressions_90d",
                          "days_since_last_update", "ctr"]]
print(top20.to_string())

              content_id  score          reason_code              action  is_declining  impressions_90d  days_since_last_update   ctr
0   content_cf56e2e2e282  61678  stale_and_declining  review_for_refresh             1            61678                     194  0.15
1   content_7368877ea310  59472  stale_and_declining  review_for_refresh             1            59472                     194  0.13
2   content_1bfaa38ff26c  25715  stale_and_declining  review_for_refresh             1            25715                     194  0.23
3   content_0a91db491d14  13299  stale_and_declining  review_for_refresh             1            13299                     193  0.49
4   content_5feee3994adb   7812  stale_and_declining  review_for_refresh             1             7812                     194  0.01
5   content_c2d929d83eaa   7558  stale_and_declining  review_for_refresh             1             7558                     193  0.20
6   content_b16bd7307b39   4590  stale_and_declining  review_f

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review:

1. content_cf56e2e2e282 — review_for_refresh, stale_and_declining. High
confidence: huge impressions (61,678) + stale + low CTR (0.15) + already
declining. Wrong if: the low CTR is a category norm (e.g. an FAQ page),
not an actual problem.

2. content_7368877ea310 — review_for_refresh, stale_and_declining. High
confidence: same pattern as #1 (high impressions, stale, low CTR). Wrong
if: recent algorithm/SERP change unrelated to content quality.

3. content_1bfaa38ff26c — review_for_refresh, stale_and_declining. Medium
confidence: still high impressions but CTR (0.23) less extreme. Wrong if:
this is normal CTR for its position tier.

4. content_0a91db491d14 — review_for_refresh, stale_and_declining. Low
confidence: CTR (0.49) is actually decent — declining despite good CTR
suggests position/volume loss, not a content problem my rule assumes.
Wrong if: the real issue is ranking position, which a refresh won't fix.

5. content_5feee3994adb — review_for_refresh, stale_and_declining. High
confidence: CTR near zero (0.01) with meaningful impressions — a real
CTR problem. Wrong if: impressions are from an irrelevant/broad query.

6. content_b16bd7307b39 — review_for_refresh, stale_and_declining.
Caution: CTR is exactly 0.00 with 4,590 impressions — worth checking if
this is a real measurement or a data artifact before acting.

7. content_fe16a55cd13d — review_for_refresh, stale_and_declining. Medium
confidence: CTR (0.33) isn't unusually low. Wrong if: this page is fine
and only flagged because it's old.

8. content_ecb6215e79fd — review_for_refresh, stale_and_declining. Low
confidence: CTR (0.38) is reasonable — flagged mainly for staleness, not
a clear CTR/decline problem.

9. content_928af3e22c80 — review_for_refresh, stale_and_declining. Medium
confidence: lower impressions (1,697) than the pages above, so less
volume at stake even if the pattern holds.

10. content_e3ff1b093148 — review_for_refresh, stale_and_declining.
Medium confidence: moderate CTR (0.28), lower volume — a smaller-impact
fix than the top few.

11. content_bdbec75c1148 — review_for_refresh, stale_but_visible. Notably
NOT declining (is_declining=0) — flagged purely for staleness+visibility.
This is the clearest case where my rule could be wrong: a stale page
that's actually still performing fine.

12. content_7f116ae1f6f5 — review_for_refresh, stale_and_declining.
Medium confidence: older still (301 days), decent CTR (0.42) — likely a
volume/position issue more than a content-quality one.

13. content_77d4d5930e5e — review_for_refresh, stale_and_declining.
Medium confidence: modest impressions and CTR, lower-priority than the
top 10.

14. content_72496874f806 — review_for_refresh, stale_and_declining.
Similar to #12 — old (301 days), reasonable CTR, lower volume.

15. content_6226ee6adc91 — review_for_refresh, stale_and_declining. Low
volume (545 impressions) — even if correct, low overall impact.

16. content_074ba6ead17b — review_for_refresh, stale_and_declining. CTR
is 0.00 again with low volume — same data-quality caution as #6.

17. content_e3393b0b5359 — monitor, no_flag. Score is 0 — tied with
several others at the bottom of "worth reviewing," ranked here somewhat
arbitrarily. Not actually declining.

18. content_f21e4a700f92 — monitor, low_ctr_at_position. This one IS
declining (is_declining=1) but got scored 0 and pushed to "monitor"
because it's not stale (only 20 days old) — a clear rule blind spot:
my staleness threshold misses recently-updated pages that are already
declining.

19. content_ccaae106ecb6 — monitor, no_flag. This is the most concerning
miss: 43,654 impressions (higher than most of my "review" list) but only
104 days old, so it falls under my 180-day stale threshold and gets
ignored entirely. A high-traffic page like this probably deserves
attention regardless of staleness.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: #11 (content_bdbec75c1148) is the weakest — it's flagged for
staleness+visibility but is NOT actually declining, showing my rule can
false-positive on healthy-but-old pages. #6, #16 are also weak due to
CTR=0.00, which may be a data artifact rather than a real signal.

Leakage check: My rule uses only days_since_last_update, impressions_90d,
and ctr — all observable before any decision point. No product-computed
flags (health_score, priority_score, action_type) were used, since those
aren't shipped in this dataset anyway. No future-window data was used —
is_declining_label itself is a same-window proxy, not something from a
future period, so there's no forward-leakage risk in this baseline.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.